# 02 · The scroll question

A new question with two numbers, where bigger always means "more likely yes". This notebook only sets it up and finds the best line by search; the next ones teach a neuron to find a line on its own.

**The question:** Mario is left of the line where the screen starts to follow him. Will the screen move within the next half second (30 frames)?

In 00 you found the line: the screen only scrolls once Mario's position byte reaches **80** (5 blocks; a block is 16 pixels). So the answer depends on how far he still has to go, and how fast he's going.

| address | what | used as |
|---|---|---|
| `0x0755` | Mario's position on screen, as the scroll code reads it (pixels) | ÷ 16: **blocks** from the left edge |
| `0x0057` | horizontal speed (signed sixteenths of a pixel a frame) | ÷ 16: **pixels a frame** to the right (walking tops out at 1.5, running 2.5) |
| `0x0775` | how far the screen scrolled this frame | **the answer key**: yes if it's non-zero on any of the next 30 frames |
| `0x000E` | player state | `08` = playing |

**What 002 says** (`greenstone/docs/series/games/002.md`; learned on sessions 1, 2, 4, 5, scored on 3 and 6):

- Many moments sit on the same spot (same position, same speed) but have different answers, because the neuron can't see what the player presses next. So no line gets everything right: the best any rule could do on the learning moments, even giving every spot its own answer, is **95%**.
- The best straight line scores **91%**: `blocks + 1.61 × speed > 3.98`. Standing still, it says yes from about 4 blocks; at walking speed, from about 1.5.

The plan: record → watch → make the moments → look → spots that hold both answers → build the neuron → dials → search → read the line.

In [ ]:
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ipywidgets import interact, IntSlider, FloatSlider

import greenstone as gs

STATE, POSITION, SPEED, SCROLL, PAGE = 0x000E, 0x0755, 0x0057, 0x0775, 0x00B5
LINE = 80   # the screen only follows Mario from here (00, exercise 2)
SOON = 30   # frames: half a second
DATA = Path("data")  # recordings are cached here, shared with 01 (git ignores data/)


def show(*pictures, titles=()):
    fig, ax = plt.subplots(1, len(pictures), figsize=(4 * len(pictures), 3.75))
    for a, p, t in zip(np.atleast_1d(ax), pictures, list(titles) + [None] * len(pictures)):
        a.imshow(p); a.axis("off")
        if t:
            a.set_title(t, fontsize=9)
    plt.show()

## The random player

The same player as 01 (copied here so this notebook stands alone): button combinations, mostly running right and jumping, held for 4 frames plus a random extra; play starts over from 1-1 when Mario dies. If 01 already recorded a session, it loads from `data/`.

In [ ]:
COMBOS = [  # (buttons, how often)
    (gs.RIGHT, 0.30),
    (gs.RIGHT | gs.A, 0.25),
    (gs.RIGHT | gs.B, 0.10),
    (gs.RIGHT | gs.A | gs.B, 0.10),
    (0, 0.10),
    (gs.LEFT, 0.05),
    (gs.A, 0.10),
]
CHECK = 600


def random_player(seed, frames):
    """A button for every frame: combinations held for 4 + a geometric number of frames (mean 16)."""
    rng = np.random.default_rng(seed)
    buttons, weights = zip(*COMBOS)
    weights = np.array(weights) / sum(weights)
    track = []
    while len(track) < frames:
        held = buttons[rng.choice(len(buttons), p=weights)]
        track += [held] * (4 + rng.geometric(1 / 17) - 1)
    return np.array(track[:frames], np.uint8)


def died(ram):
    return np.isin(ram[:, STATE], (0x06, 0x0B)) | (ram[:, PAGE] >= 2)


def record(seed, frames=18_000):
    """Memory after every frame, the buttons, the frames he died on, and a saved moment every 600 frames."""
    player = hashlib.sha1(repr(COMBOS).encode()).hexdigest()[:8]
    path = DATA / f"smb1-random-{player}-{seed}-{frames}.npz"
    if path.exists():
        z = np.load(path)
        return dict(ram=z["ram"], buttons=z["buttons"], restarts=list(z["restarts"]),
                    checkpoints=list(zip(z["check_frames"], z["check_states"])))
    nes = gs.boot("smb1")
    start = nes.save()
    buttons = random_player(seed, frames)
    rams, restarts, checkpoints = [], [], []
    f = 0
    while f < frames:
        checkpoints.append((f, np.frombuffer(nes.save(), np.uint8)))
        chunk = nes.play(buttons[f : f + CHECK], pictures=False)
        dead = np.flatnonzero(died(chunk))
        if len(dead):
            chunk = chunk[: dead[0] + 1]
            nes.load(start)
            restarts.append(f + dead[0])
        rams.append(chunk)
        f += len(chunk)
    ram = np.concatenate(rams)
    DATA.mkdir(exist_ok=True)
    frames_, states = zip(*checkpoints)
    np.savez_compressed(path, ram=ram, buttons=buttons, restarts=np.array(restarts, int),
                        check_frames=np.array(frames_), check_states=np.stack(states))
    return dict(ram=ram, buttons=buttons, restarts=restarts, checkpoints=checkpoints)


def picture(session, f):
    """The screen showing frame f's memory (the picture runs one frame behind)."""
    cf, state = [c for c in session["checkpoints"] if c[0] <= f][-1]
    nes = gs.boot("smb1")
    nes.load(state.tobytes())
    _, pics = nes.play(session["buttons"][cf : f + 2], pictures=True)
    return pics[-1]

Six five-minute sessions. Each new one takes a minute or two to record.

In [ ]:
sessions = {seed: record(seed) for seed in range(1, 7)}
LEARN, SCORE = (1, 2, 4, 5), (3, 6)

## Watch it

The first thirty seconds of session 1. The red line is 80: the scroll byte stays at zero until the position reaches it. Scrub, and watch the screen start to follow him.

In [ ]:
s1 = sessions[1]
r = s1["ram"][:1800]
fig, ax = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
ax[0].plot(r[:, POSITION]); ax[0].axhline(LINE, color="red", lw=0.8); ax[0].set_ylabel("position (px)")
ax[1].plot(r[:, SPEED].astype(np.int8) / 16); ax[1].set_ylabel("speed (px/frame)")
ax[2].plot(r[:, SCROLL]); ax[2].set_ylabel("scrolled (px)")
ax[2].set_xlabel("frame"); plt.show()


def look(f):
    m = r[f]
    soon = (s1["ram"][f + 1 : f + SOON + 1, SCROLL] != 0).any()
    show(picture(s1, f), titles=[f"frame {f}: position {m[POSITION]} ({m[POSITION] / 16:.2f} blocks), "
                                 f"speed {np.int8(m[SPEED]) / 16:.2f}\nscreen moves within 30 frames? {'yes' if soon else 'no'}"])

interact(look, f=IntSlider(0, 0, len(r) - 1));

## The moments

Not every frame is a fair question. A frame is a **moment** if:

- Mario is **left of the line** (position below 80). Past it, the answer is just "is he walking right?"
- he's **not walking left** (speed 0 or more). Walking left, the screen can't move until he turns round, and his speed would be negative, breaking "bigger means more yes".
- the game is **in play** for the whole next 30 frames, and he **doesn't die** inside them (a fresh start isn't the screen moving).

Each moment becomes two numbers, **blocks** (position ÷ 16) and **pixels a frame** (speed ÷ 16), and one answer: does the scroll byte move on any of the next 30 frames?

(Why divide by 16: the numbers read as things you can picture: "2 blocks from the edge, walking at 1.5". It also matters for learning, in a later notebook.)

In [ ]:
def moments(session):
    """Each moment as (blocks, pixels a frame), whether the screen moved within SOON frames, and its frame."""
    ram = session["ram"]
    n = len(ram)
    play = (ram[:, STATE] == 0x08) & ~died(ram)
    moved = ram[:, SCROLL] != 0
    restart = np.zeros(n, bool)
    restart[session["restarts"]] = True
    speed = ram[:, SPEED].astype(np.int8)
    keep = [i for i in range(n - SOON)
            if play[i] and ram[i, POSITION] < LINE and speed[i] >= 0
            and play[i + 1 : i + SOON + 1].all() and not restart[i + 1 : i + SOON + 1].any()]
    keep = np.array(keep)
    soon = np.array([moved[i + 1 : i + SOON + 1].any() for i in keep])
    x = np.stack([ram[keep, POSITION] / 16, speed[keep] / 16], axis=1)
    return x, soon, keep


data = {seed: moments(s) for seed, s in sessions.items()}
for seed, (x, y, _) in data.items():
    print(f"session {seed}: {len(x)} moments, screen moved soon on {y.mean():.0%}")

x_learn = np.concatenate([data[s][0] for s in LEARN]); y_learn = np.concatenate([data[s][1] for s in LEARN])
x_score = np.concatenate([data[s][0] for s in SCORE]); y_score = np.concatenate([data[s][1] for s in SCORE])
print(f"\nlearn on {len(x_learn)} moments, score on {len(x_score)}")
print(f"always say 'no': {(~y_learn).mean():.1%} on the learning moments")

## Look

Each dot is a moment from the learning sessions. Many moments sit on exactly the same spot, so the dots are shaken a little to show the piles. What line would you draw?

In [ ]:
shake = np.random.default_rng(0).uniform(-0.03, 0.03, x_learn.shape)


def scatter(x, y, ax=None):
    ax = ax or plt.gca()
    p = x + shake[: len(x)] if len(x) <= len(shake) else x
    ax.scatter(p[~y, 0], p[~y, 1], s=2, alpha=0.25, label="screen doesn't move within 30")
    ax.scatter(p[y, 0], p[y, 1], s=2, alpha=0.25, label="screen moves within 30")
    ax.set_xlabel("blocks from the left edge"); ax.set_ylabel("pixels a frame to the right")
    ax.set_xlim(-0.2, 5.2); ax.set_ylim(-0.2, 2.8)
    ax.legend(markerscale=6, loc="upper left")


plt.figure(figsize=(8, 5)); scatter(x_learn, y_learn); plt.show()

## Spots that hold both answers

**Exercise 1.** Call two moments the same **spot** if they have the same two numbers.

1. How many spots do the learning moments sit on?
2. How many spots hold both answers?
3. Suppose a rule could give every spot its own answer, any shape at all. The best it can do is give each spot whichever answer most of its moments have. What share of the learning moments would that get right? That's the ceiling for *any* neuron reading these two numbers.

(002, with its own sessions: 1,313 spots, 232 holding both, 95%. Yours won't match exactly. Hint: `np.unique(x_learn, axis=0, return_inverse=True)`.)

In [ ]:
# Exercise 1

Why does one spot hold both answers? Here are two moments from the busiest mixed spot: same position, same speed, different futures. Below each one is what the player held over the next 30 frames, which the neuron never sees.

In [ ]:
where_learn = [(s, f) for s in LEARN for f in data[s][2]]
spots, which = np.unique(x_learn, axis=0, return_inverse=True)
which = which.ravel()
yes_per = np.bincount(which, y_learn, len(spots)); n_per = np.bincount(which, minlength=len(spots))
mixed = np.flatnonzero((yes_per > 0) & (yes_per < n_per))
busiest = mixed[np.argsort(-n_per[mixed])]


def held_right(seed, f):
    return ((sessions[seed]["buttons"][f + 1 : f + SOON + 1] & gs.RIGHT) != 0).mean()


def pair(k=0, i=0):
    spot = busiest[k]
    at = np.flatnonzero(which == spot)
    yes, no = at[y_learn[at]], at[~y_learn[at]]
    a, b = where_learn[yes[i % len(yes)]], where_learn[no[i % len(no)]]
    print(f"spot {spots[spot][0]:.4f} blocks, {spots[spot][1]:.4f} px/frame: "
          f"{n_per[spot]} moments, {int(yes_per[spot])} yes")
    show(picture(sessions[a[0]], a[1]), picture(sessions[b[0]], b[1]),
         titles=[f"session {a[0]} frame {a[1]}: screen moved\nRight held {held_right(*a):.0%} of the next 30",
                 f"session {b[0]} frame {b[1]}: didn't\nRight held {held_right(*b):.0%} of the next 30"])

interact(pair, k=IntSlider(0, 0, min(len(busiest), 30) - 1, description="spot"),
         i=IntSlider(0, 0, 20, description="example"));

## The neuron, from scratch

Same neuron as 01, with two inputs: each input times its weight, added up, plus the bias. It says "yes" if the total is above zero.

$$\text{total} = w_1 \cdot \text{blocks} + w_2 \cdot \text{speed} + b$$

**Exercise 2a.** Write it the long way, as *Neural Networks from Scratch* does in chapter 2: plain Python, lists, a loop over the inputs. It takes one moment's inputs (a list of two numbers), the weights (a list of two) and the bias, and returns the **total**.

**Exercise 2b.** Now all the moments at once with NumPy: `x` is an array of shape `(moments, 2)`, `w` an array of 2. Return every total, shape `(moments,)`. (One `@` does it.)

In [ ]:
def neuron(inputs, weights, bias):
    # Exercise 2a
    raise NotImplementedError


def totals(x, w, b):
    # Exercise 2b
    raise NotImplementedError

In [ ]:
# The two should agree.
w, b = np.array([1.0, 1.6]), -4.0
long_way = [neuron(list(m), list(w), b) for m in x_learn[:200]]
assert np.allclose(long_way, totals(x_learn[:200], w, b)), "they disagree"
print("they agree")


def accuracy(x, y, w, b):
    return ((totals(x, w, b) > 0) == y).mean()

## Dials

The dashed line is where the total is zero: yes on one side, no on the other. Try making each weight 0 (one number alone), or negative.

In [ ]:
def dials(w1=1.0, w2=1.0, b=-3.0):
    w = np.array([w1, w2])
    plt.figure(figsize=(8, 5)); scatter(x_learn, y_learn)
    d = np.array([-0.2, 5.2])
    if w2 != 0:
        plt.plot(d, -(w1 * d + b) / w2, "k--")
    elif w1 != 0:
        plt.axvline(-b / w1, color="k", ls="--")
    plt.title(f"{accuracy(x_learn, y_learn, w, b):.1%} right on the learning moments")
    plt.show()

interact(dials, w1=FloatSlider(1, min=-3, max=3, step=0.05), w2=FloatSlider(1, min=-3, max=3, step=0.05),
         b=FloatSlider(-3, min=-10, max=10, step=0.05));

## The best line, by search

**Exercise 3.** Find the line that gets the most learning moments right, by trying lots of them. Then score it, once, on sessions 3 and 6. (002: 91%.)

A hint, if you want one: a line is a direction and a cut. Pick a direction `w = (cos θ, sin θ)`; then every moment's `w · x` is one number, and finding the best cut on one number is 01's exercise 2. Try a few hundred directions.

**Exercise 4.** Read your line. Divide through by `w1` to write it as `blocks + k × speed > c`.
- Standing still, from how many blocks does it say yes? (002: about 4.)
- Walking (1.5 pixels a frame)? (002: about 1.5.)
- Does that fit what you'd expect from 80 pixels being 5 blocks and half a second being 30 frames?
- If your line isn't 002's, find out why. Score 002's line (`blocks + 1.61 × speed > 3.98`) on your moments too: where do the two disagree, and what's going on in the pictures there? (The busiest mixed spot above is a clue.)

**Exercise 5.** Compare: always "no", your line, and exercise 1's ceiling. How much of the gap between your line and the ceiling is the line's straightness, and how much is the ceiling below 100%?

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4

In [ ]:
# Exercise 5

## Where the line is wrong

Put your line in `W, B` and scrub the scoring moments it gets wrong.

In [ ]:
W, B = np.array([1.0, 1.6]), -4.0  # your line from exercise 3

where_score = [(s, f) for s in SCORE for f in data[s][2]]
wrong = np.flatnonzero((totals(x_score, W, B) > 0) != y_score)
print(f"{len(wrong)} wrong of {len(x_score)} ({1 - len(wrong) / len(x_score):.1%} right)")


def mistake(i):
    j = wrong[i]
    seed, f = where_score[j]
    said = "yes" if totals(x_score[j : j + 1], W, B)[0] > 0 else "no"
    show(picture(sessions[seed], f),
         titles=[f"session {seed} frame {f}: {x_score[j, 0]:.2f} blocks, {x_score[j, 1]:.2f} px/frame\n"
                 f"line says {said}; Right held {held_right(seed, f):.0%} of the next 30"])

interact(mistake, i=IntSlider(0, 0, max(len(wrong) - 1, 0)));

## Play

1. **The third number.** 002 found that adding "share of the next 30 frames Right was held" as a third input lifts a straight line from 90% to 94%. Try it. Why can't a real neuron use that input?
2. **A different window.** Change `SOON` to 15 or 60 and rerun from "The moments". How do the best line and the ceiling move?
3. **A steadier player.** Give `COMBOS` more Right. Does the ceiling go up?